# Azure DevOps Features Analysis

This notebook reads the Azure DevOps backlog data and creates a comprehensive DataFrame showing all features with their characteristics.

## 1. Import libraries and read data
### Libraries

In [ ]:
import json
import pandas as pd
from pathlib import Path
from datetime import date

### Refresh Backlog Data
Fetch the latest data from Azure DevOps and transform it into a hierarchical structure.  

Runtime ~ 1.15 min

In [ ]:
%run ../src/scripts/fetch_data_science_pool_backlog.py
%run ../src/scripts/backlog_hierarchical_transformer.py

## 2. Load JSON Data

In [ ]:
# Define the path to the JSON file using today's date
today = date.today().strftime('%Y%m%d')
data_file = Path(f'../azure_devops_data/backlog_simplified_{today}.json')

# Load the JSON data
with open(data_file, 'r', encoding='utf-8') as f:
    backlog_data = json.load(f)

# Display metadata
print("Metadata:")
print(f"Organization: {backlog_data['metadata']['organization']}")
print(f"Total work items: {backlog_data['metadata']['total_work_items']}")
print(f"Data fetched at: {backlog_data['metadata']['source_fetched_at']}")
print("\nStatistics by type:")
for work_type, count in backlog_data['statistics']['by_type'].items():
    print(f"  {work_type}: {count}")

### Define parameters

In [ ]:
target_hours_2026 = 1991

## 3. Gather Data on User Stories
### Get all User Stories and their parent Feature and Epic information

We need to traverse the nested structure recursively to find all User Stories and include ALL information from their parent Features and Epics. \
Note: some Epics (and/or features) might not have children (User Stories), so we handle that case as well.

In [ ]:
EXCLUDE_KEYS = {'children', 'type'}

def extract_user_stories(items, parent_epic=None, parent_feature=None):
    """
    Recursively extract all User Stories from backlog items,
    including all available fields from their parent Feature and Epic.
    Fields are prefixed: us_ (user story), f_ (feature), e_ (epic).

    Epics or Features with no descendant User Stories are still included
    as rows with missing us_/f_ fields.
    """
    user_stories = []

    for item in items:
        # Track parent Epic and Feature as we descend
        current_epic = item if item.get('type') == 'Epic' else parent_epic
        current_feature = item if item.get('type') == 'Feature' else parent_feature

        # If current item is a User Story, capture it with all available fields
        if item.get('type') == 'User Story':
            story_data = {f'us_{k}': v for k, v in item.items() if k not in EXCLUDE_KEYS}
            if current_feature:
                story_data.update({f'f_{k}': v for k, v in current_feature.items() if k not in EXCLUDE_KEYS})
            if current_epic:
                story_data.update({f'e_{k}': v for k, v in current_epic.items() if k not in EXCLUDE_KEYS})
            user_stories.append(story_data)

        # Recursively process children
        children_stories = []
        if item.get('children'):
            children_stories = extract_user_stories(
                item['children'],
                parent_epic=current_epic,
                parent_feature=current_feature
            )
            user_stories.extend(children_stories)

        # Epic or Feature with no descendant User Stories: add placeholder row
        if item.get('type') in ('Epic', 'Feature') and not children_stories:
            row = {}
            if item.get('type') == 'Feature':
                row.update({f'f_{k}': v for k, v in item.items() if k not in EXCLUDE_KEYS})
                if current_epic:
                    row.update({f'e_{k}': v for k, v in current_epic.items() if k not in EXCLUDE_KEYS})
            elif item.get('type') == 'Epic':
                row.update({f'e_{k}': v for k, v in item.items() if k not in EXCLUDE_KEYS})
            user_stories.append(row)

    return user_stories

# Extract all user stories from the backlog with Feature and Epic information
all_user_stories = extract_user_stories(backlog_data['backlog'])

print(f"Total User Stories found: {len(all_user_stories)}")
print(f"User Stories with Feature parent: {sum(1 for s in all_user_stories if s.get('f_id') is not None)}")
print(f"Features with Epic parent: {sum(1 for s in all_user_stories if s.get('e_id') is not None and s.get('f_id') is not None)}")
print(f"Rows without User Story (Epic/Feature only): {sum(1 for s in all_user_stories if s.get('us_id') is None)}")

# Convert the list of user stories to a DataFrame
user_stories_df = pd.DataFrame(all_user_stories)

## 4. Process Data

### 4.1 Aggregate hours-spent to Epic level

In [ ]:
# Create a epic_df DataFrame with one row per Epic, including some Epic-level fields
# It should include the summed user story spent hours (us_completed_work) for each Epic.
epic_df = user_stories_df.groupby('e_id').agg(
    e_name=('e_title', 'first'),
    e_start_date=('e_start_date', 'first'),
    e_end_date=('e_target_date', 'first'),
    e_hours_budgeted=('e_hours', 'first'),
    e_hours_spent=('us_completed_work', 'sum'),
    e_project_owner=('e_project_owner', 'first'),
    e_project_area=('e_project_area', 'first'),
).reset_index()

# Create a lookup of spent hours by (epic, year, quarter) based on user story closed_date
# This ensures spent hours are attributed to the year the work was actually completed
# If closed_date is missing, use created_date as fallback
us_with_year = user_stories_df[user_stories_df['us_id'].notna()].copy()
us_with_year['us_closed_dt'] = pd.to_datetime(us_with_year['us_closed_date'], errors='coerce')
us_with_year['us_created_dt'] = pd.to_datetime(us_with_year['us_created_date'], errors='coerce')
# Use closed_date if available, otherwise fall back to created_date
us_with_year['us_effective_dt'] = us_with_year['us_closed_dt'].fillna(us_with_year['us_created_dt'])
us_with_year['us_year'] = us_with_year['us_effective_dt'].dt.year
us_with_year['us_quarter'] = us_with_year['us_effective_dt'].dt.quarter

spent_by_epic_year = us_with_year.groupby(['e_id', 'us_year'])['us_completed_work'].sum().reset_index()
spent_by_epic_year.columns = ['e_id', 'year', 'hours_spent']

# Also create a lookup by (epic, year, quarter)
spent_by_epic_year_quarter = us_with_year.groupby(['e_id', 'us_year', 'us_quarter'])['us_completed_work'].sum().reset_index()
spent_by_epic_year_quarter.columns = ['e_id', 'year', 'quarter', 'hours_spent']

### 4.2 Data cleaning and processing
Remove unnecessary fields, handle missing values, and ensure all relevant information is included in the final DataFrame.

In [ ]:
# Set budgeted hours to 0 is missing (NaN) and ensure it's numeric
epic_df['e_hours_budgeted'] = pd.to_numeric(epic_df['e_hours_budgeted'], errors='coerce').fillna(0)

# Set project owner and area to '' if missing
epic_df['e_project_owner'] = epic_df['e_project_owner'].fillna('')
epic_df['e_project_area'] = epic_df['e_project_area'].fillna('')

# Set e_id to integer
epic_df['e_id'] = pd.to_numeric(epic_df['e_id'], errors='coerce').fillna(0).astype(int)

### 4.3 Data enhancement: Epic per fiscal year
Extract the fiscal year from the 'e_start_date' field and create a new column 'e_year' in the DataFrame.
If an Epic spans multiple years, we will create a separate entry for each year it spans, with the hours allocated proportionally based on the duration in each year.

In [ ]:
# Quarter date boundaries helper
def quarter_bounds(year, quarter):
    """Return (start, end) Timestamps for a given year and quarter."""
    q_starts = {1: (1, 1), 2: (4, 1), 3: (7, 1), 4: (10, 1)}
    q_ends   = {1: (3, 31), 2: (6, 30), 3: (9, 30), 4: (12, 31)}
    return pd.Timestamp(year, *q_starts[quarter]), pd.Timestamp(year, *q_ends[quarter])

# Create lists to store expanded epic records at year and year-quarter level
epic_year_records = []
epic_yq_records = []

for _, epic in epic_df.iterrows():
    # Parse dates and convert from UTC to local timezone before removing tz info
    start_date = pd.to_datetime(epic['e_start_date'], errors='coerce')
    end_date = pd.to_datetime(epic['e_end_date'], errors='coerce')
    
    # Convert to local timezone (CET/CEST) before removing timezone info
    if start_date is not pd.NaT and start_date.tz is not None:
        start_date = start_date.tz_convert('Europe/Amsterdam').tz_localize(None)
    if end_date is not pd.NaT and end_date.tz is not None:
        end_date = end_date.tz_convert('Europe/Amsterdam').tz_localize(None)
    
    # Get actual spent hours per year and per year-quarter for this epic
    epic_spent = spent_by_epic_year[spent_by_epic_year['e_id'] == epic['e_id']]
    spent_lookup = {int(k): v for k, v in zip(epic_spent['year'], epic_spent['hours_spent'])}

    epic_spent_q = spent_by_epic_year_quarter[spent_by_epic_year_quarter['e_id'] == epic['e_id']]
    spent_q_lookup = {(int(r['year']), int(r['quarter'])): r['hours_spent'] for _, r in epic_spent_q.iterrows()}
    
    # Common fields for all records of this epic
    common_fields = {
        'e_id': epic['e_id'],
        'e_name': epic['e_name'],
        'e_project_owner': epic['e_project_owner'],
        'e_project_area': epic['e_project_area'],
        'e_start_date': epic['e_start_date'],
        'e_end_date': epic['e_end_date'],
    }
    
    # If dates are missing, assign to a single year with NaN quarter
    if pd.isna(start_date):
        epic_year_records.append({
            **common_fields,
            'e_year': pd.NA,
            'e_hours_budgeted': epic['e_hours_budgeted'],
            'e_hours_spent': epic['e_hours_spent'],
        })
        epic_yq_records.append({
            **common_fields,
            'e_year': pd.NA,
            'e_quarter': pd.NA,
            'e_hours_budgeted': epic['e_hours_budgeted'],
            'e_hours_spent': epic['e_hours_spent'],
        })
        continue
    
    # If end date is missing, use start date year only
    if pd.isna(end_date):
        end_date = start_date
    
    # Get start and end years
    start_year = int(start_date.year)
    end_year = int(end_date.year)
    
    # Calculate total duration in days
    total_days = (end_date - start_date).days + 1
    if total_days <= 0:
        total_days = 1
    
    # Collect all years: epic date range + any years with actual spent hours
    all_years = set(range(start_year, end_year + 1)) | set(spent_lookup.keys())
    # Also include years from quarter-level spent data
    all_years |= {y for y, q in spent_q_lookup.keys()}
    
    for year in sorted(all_years):
        # --- Year-level record (unchanged logic) ---
        if start_year <= year <= end_year:
            year_start = max(start_date, pd.Timestamp(year, 1, 1))
            year_end = min(end_date, pd.Timestamp(year, 12, 31))
            days_in_year = (year_end - year_start).days + 1
            proportion = days_in_year / total_days
            budgeted_year = epic['e_hours_budgeted'] * proportion
        else:
            budgeted_year = 0
        
        spent_year = spent_lookup.get(year, 0)
        
        epic_year_records.append({
            **common_fields,
            'e_year': year,
            'e_hours_budgeted': budgeted_year,
            'e_hours_spent': spent_year,
        })
        
        # --- Quarter-level records ---
        # Determine which quarters to include for this year
        q_with_spent = {q for (y, q) in spent_q_lookup.keys() if y == year}
        q_in_range = set()
        if start_year <= year <= end_year:
            for q in range(1, 5):
                q_start, q_end = quarter_bounds(year, q)
                # Epic overlaps this quarter if epic period intersects the quarter
                if start_date <= q_end and end_date >= q_start:
                    q_in_range.add(q)
        all_quarters = q_in_range | q_with_spent
        
        for q in sorted(all_quarters):
            q_start, q_end = quarter_bounds(year, q)
            # Budgeted: proportional overlap of epic with this quarter
            if start_year <= year <= end_year and q in q_in_range:
                overlap_start = max(start_date, q_start)
                overlap_end = min(end_date, q_end)
                overlap_days = (overlap_end - overlap_start).days + 1
                budgeted_q = epic['e_hours_budgeted'] * max(overlap_days, 0) / total_days
            else:
                budgeted_q = 0
            
            spent_q = spent_q_lookup.get((year, q), 0)
            
            epic_yq_records.append({
                **common_fields,
                'e_year': year,
                'e_quarter': q,
                'e_hours_budgeted': budgeted_q,
                'e_hours_spent': spent_q,
            })

# Create the DataFrames
epic_year_df = pd.DataFrame(epic_year_records)
epic_yq_df = pd.DataFrame(epic_yq_records)

# Display summary
print(f"Original Epics: {epic_df['e_id'].nunique()}")
print(f"Total Epic-Year records: {len(epic_year_df)}")
print(f"Total Epic-Year-Quarter records: {len(epic_yq_df)}")

## 5. Create a KPI Dashboard
Showing the following KPIs:
- Total number of Epics per year, including budgeted hours (+target) and spent hours
- Ability to toggle between years

In [ ]:
# Aggregate by year: count epics and sum budgeted hours
year_agg = epic_year_df.groupby('e_year').agg(
    epic_count=('e_id', 'count'),
    hours_budgeted=('e_hours_budgeted', 'sum'),
    hours_spent=('e_hours_spent', 'sum')
).reset_index()

# Display the summary table
print("\nSummary: Epics per Year")
print(year_agg.to_string(index=False))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Generic work item URL that works for all types (Epic, Feature, User Story)
DEVOPS_BASE_URL = "https://dev.azure.com/HogeschoolUtrecht/Data%20Science%20Pool/_workitems/edit/"

# Get available years (drop NaN)
available_years = sorted(epic_year_df['e_year'].dropna().unique().astype(int))

year_dropdown = widgets.Dropdown(
    options=available_years,
    value=available_years[-1] if available_years else None,
    description='Year:',
    style={'description_width': 'initial'}
)

quarter_dropdown = widgets.Dropdown(
    options=[("All Q's", 0), ('Q1', 1), ('Q2', 2), ('Q3', 3), ('Q4', 4)],
    value=0,
    description='Quarter:',
    style={'description_width': 'initial'}
)

output = widgets.Output()

MAX_NAME_LEN = 55

def update_dashboard(*_):
    with output:
        clear_output(wait=True)
        year = year_dropdown.value
        quarter = quarter_dropdown.value

        if quarter == 0:
            filtered = epic_year_df[epic_year_df['e_year'] == year]
            period_label = str(year)
            target_hours = target_hours_2026
        else:
            filtered = epic_yq_df[
                (epic_yq_df['e_year'] == year) & (epic_yq_df['e_quarter'] == quarter)
            ]
            period_label = f"{year} Q{quarter}"
            target_hours = target_hours_2026 / 4

        n_epics = filtered['e_id'].nunique()
        total_budgeted = filtered['e_hours_budgeted'].sum()
        total_budgeted_spent = filtered.loc[filtered['e_hours_budgeted'] > 0, 'e_hours_spent'].sum()

        # Calculate KPI percentage
        kpi_percentage = (total_budgeted / target_hours * 100) if target_hours > 0 else 0
        kpi_color = '#28a745' if kpi_percentage >= 100 else '#dc3545'  # green if >= 100%, red otherwise

        # Build HTML table with clickable Epic IDs, sorted by budgeted hours descending
        rows_html = ""
        display_df = filtered.drop_duplicates(subset='e_id').sort_values('e_hours_budgeted', ascending=False)
        for _, row in display_df.iterrows():
            eid = int(row['e_id'])
            name = row['e_name']
            if len(name) > MAX_NAME_LEN:
                name = name[:MAX_NAME_LEN - 3] + '...'
            link = f'<a href="{DEVOPS_BASE_URL}{eid}" target="_blank">{eid}</a>'
            owner = row.get('e_project_owner', '') or ''
            area = row.get('e_project_area', '') or ''
            rows_html += (
                f"<tr>"
                f"<td style='text-align:right; padding:4px 8px;'>{link}</td>"
                f"<td style='text-align:left; padding:4px 8px;'>{name}</td>"
                f"<td style='text-align:left; padding:4px 8px;'>{owner}</td>"
                f"<td style='text-align:left; padding:4px 8px;'>{area}</td>"
                f"<td style='text-align:right; padding:4px 8px;'>{row['e_hours_budgeted']:.1f}</td>"
                f"<td style='text-align:right; padding:4px 8px;'>{row['e_hours_spent']:.1f}</td>"
                f"</tr>\n"
            )

        html = f"""
        <h3>{period_label}</h3>
        <div style="margin-bottom: 20px;">
            <div style="display: inline-block; padding: 12px 20px; background-color: {kpi_color}; color: white; border-radius: 8px; font-size: 19px; font-weight: bold; box-shadow: 0 2px 4px rgba(0,0,0,0.2);">
                Budget vs Target: {kpi_percentage:.1f}%
            </div>
            <div style="margin-top: 10px; font-size: 14px; color: #666;">
            </div>
        </div>
        <p>Number of Epics: {n_epics}<br>
        Total Hours Budgeted: {total_budgeted:,.0f} &nbsp;|&nbsp; Target Hours: {target_hours:,.0f}<br>
        Total Budgeted Hours Spent: {total_budgeted_spent:,.0f}</p>
        <table style="border-collapse:collapse; font-family:monospace; font-size:13px;">
        <tr style="border-bottom:2px solid #333;">
            <th style="text-align:right; padding:4px 8px;">ID</th>
            <th style="text-align:left; padding:4px 8px;">Name</th>
            <th style="text-align:left; padding:4px 8px;">Owner</th>
            <th style="text-align:left; padding:4px 8px;">Area</th>
            <th style="text-align:right; padding:4px 8px;">Budgeted</th>
            <th style="text-align:right; padding:4px 8px;">Spent</th>
        </tr>
        {rows_html}
        </table>
        """
        display(HTML(html))

year_dropdown.observe(update_dashboard, names='value')
quarter_dropdown.observe(update_dashboard, names='value')

# Trigger initial display
update_dashboard()

display(widgets.HBox([year_dropdown, quarter_dropdown]), output)

## 6. Open User Stories with Hours Spent
User stories that have completed work logged but are not yet closed.

In [ ]:
# Filter: user stories with hours spent but not closed
open_with_hours = user_stories_df[
    (user_stories_df['us_id'].notna()) &
    (user_stories_df['us_completed_work'].fillna(0) > 0) &
    (user_stories_df['us_state'] != 'Closed')
].copy()

n_stories = len(open_with_hours)
total_spent = open_with_hours['us_completed_work'].sum()

MAX_TITLE_LEN = 60

# Build HTML table rows sorted by hours spent descending
rows_html = ""
for _, row in open_with_hours.sort_values('us_completed_work', ascending=False).iterrows():
    us_id = int(row['us_id'])
    title = row.get('us_title', '') or ''
    if len(title) > MAX_TITLE_LEN:
        title = title[:MAX_TITLE_LEN - 3] + '...'
    link = f'<a href="{DEVOPS_BASE_URL}{us_id}" target="_blank">{us_id}</a>'
    state = row.get('us_state', '') or ''
    spent = row.get('us_completed_work', 0) or 0
    assigned = row.get('us_assigned_to', '') or ''
    feature = row.get('f_title', '') or ''
    if len(feature) > 40:
        feature = feature[:37] + '...'
    epic = row.get('e_title', '') or ''
    if len(epic) > 40:
        epic = epic[:37] + '...'
    rows_html += (
        f"<tr>"
        f"<td style='text-align:right; padding:4px 8px;'>{link}</td>"
        f"<td style='text-align:left; padding:4px 8px;'>{title}</td>"
        f"<td style='text-align:left; padding:4px 8px;'>{state}</td>"
        f"<td style='text-align:right; padding:4px 8px;'>{spent:.1f}</td>"
        f"<td style='text-align:left; padding:4px 8px;'>{assigned}</td>"
        f"<td style='text-align:left; padding:4px 8px;'>{feature}</td>"
        f"<td style='text-align:left; padding:4px 8px;'>{epic}</td>"
        f"</tr>\n"
    )

html = f"""
<h3>Open User Stories with Hours Spent</h3>
<p>Number of User Stories: {n_stories}<br>
Total Hours Spent: {total_spent:,.1f}</p>
<table style="border-collapse:collapse; font-family:monospace; font-size:13px;">
<tr style="border-bottom:2px solid #333;">
    <th style="text-align:right; padding:4px 8px;">ID</th>
    <th style="text-align:left; padding:4px 8px;">Title</th>
    <th style="text-align:left; padding:4px 8px;">State</th>
    <th style="text-align:right; padding:4px 8px;">Spent</th>
    <th style="text-align:left; padding:4px 8px;">Assigned To</th>
    <th style="text-align:left; padding:4px 8px;">Feature</th>
    <th style="text-align:left; padding:4px 8px;">Epic</th>
</tr>
{rows_html}
</table>
"""
display(HTML(html))